<a href="https://colab.research.google.com/github/Antasey/NCAIR-DSA-Group1/blob/samuel/NCAIR_DSA_LLM_Structuring_corrected%20(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NCAIR-DSA — LLM Structuring Pipeline
### ASR Text → Full Clinical Note + Keywords → Database

**Owner:** LLM Structuring Team  
**Goal:** Take a raw ASR transcript (already transcribed from Yoruba/Igbo/Hausa audio), run it through the N-ATLaS LLM to produce a **full structured English clinical note**, extract **supporting keywords**, and save both to the patient database.

**Pipeline covered in this notebook:**
1. Setup (install packages, check GPU)
2. Load N-ATLaS LLM (local, 4-bit quantized — free on Colab T4)
3. Define the structuring prompt
4. Call the LLM + parse its JSON output (full note — the CORE deliverable)
5. Extract keywords (supporting sidebar reference — NOT the main output)
6. Combine into one patient record
7. Save to SQLite database (linked to patient ID for traceable history)
8. End-to-end test with sample ASR transcripts

> **Note:** This notebook assumes ASR transcription has already happened (that's the ASR team's module). We start from raw transcript text.

## 1. Setup — Install Packages & Check GPU

Make sure your runtime is set to GPU: **Runtime → Change runtime type → T4 GPU**

In [1]:
# Install required packages (only needs to run once per Colab session)
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.1 MB/s eta 0:00:00


In [2]:
import torch

# Confirm GPU is available — if this prints False, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


## 1b. Hugging Face Authentication (Required for Gated Models)

`NCAIR1/N-ATLaS` (and ` All NCAIR1 Language Models` if used later) are **gated** repos. Before this notebook can load them you must:

1. Log in to huggingface.co and visit the model page (e.g. https://huggingface.co/NCAIR1/N-ATLaS) and click **"Agree and access repository"** / **"Request access"**. Wait for approval if it's not instant.
2. Create an access token at https://huggingface.co/settings/tokens (Read access is enough).
3. Run the cell below and paste your token when prompted — it uses `getpass` so the token is **never printed or logged**.

**Never paste a raw token directly into a code cell or print it** — if you do, treat it as compromised and revoke it immediately from the tokens page.

In [42]:
import os
from huggingface_hub import login
from getpass import getpass

# 1. Get the token securely
hf_token = getpass("Enter your Hugging Face access token: ")

# 2. Log in the Python environment
login(token=hf_token)

# 3. CRITICAL: Export it to the system environment so !python app.py can see it
os.environ["HF_TOKEN"] = hf_token

# 4. Clear the variable from memory
del hf_token
print("Logged in to Hugging Face and exported token to environment.")

Enter your Hugging Face access token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face and exported token to environment.


## 2. Load N-ATLaS LLM (Local, 4-bit Quantized)

This downloads the model from Hugging Face directly to Google's servers (not your internet) and loads it in 4-bit precision so it fits on the free T4 GPU (16GB VRAM).

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "NCAIR1/N-ATLaS"

# 4-bit quantization config — keeps the 8B model small enough for a T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this can take a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config
)

print("Model loaded successfully.")

Loading tokenizer...
Loading model (this can take a few minutes on first run)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.


## 3. The Structuring Prompt

This prompt tells N-ATLaS exactly what to do: translate (if needed), extract four clinical fields, and return clean JSON only. This is the **core** of the LLM structuring module — tune this if output quality is inconsistent.

In [8]:
STRUCTURE_PROMPT = """You are a clinical note assistant specializing in multilingual healthcare.

Your task: Take a patient's raw speech (in Yoruba, Igbo, Hausa, or English)
and structure it into a clean, organized English clinical note.

INSTRUCTIONS:
1. If the patient spoke in Yoruba, Igbo, or Hausa, translate their meaning into clear English.
2. Extract and organize EXACTLY these four fields:
   - chief_complaint: The patient's main symptom or concern (1-2 sentences, in English)
   - duration: How long the problem has lasted (e.g., "since yesterday", "3 days", "1 week")
   - severity: How bad it is on a scale (mild, moderate, or severe)
   - history: Any other relevant details or secondary symptoms mentioned

3. CRITICAL: Use ONLY information the patient actually said. Do NOT invent or assume details.
4. If a field is not mentioned by the patient, set it to "Not mentioned".
5. Respond with ONLY valid JSON. No explanatory text before or after the JSON.

EXAMPLE:
Patient speech: "My stomach has been hurting me since yesterday, I've been vomiting too, it's very bad"
Output:
{
  "chief_complaint": "Abdominal pain with vomiting",
  "duration": "Since yesterday",
  "severity": "severe",
  "history": "Patient reports nausea accompanying the abdominal pain"
   "causes": "Patient suspects food eaten at a party",
  "effect": "Unable to go to work"
}

Now process this patient's speech:
Patient speech: {transcript}

Output (JSON only, no other text):"""

print("Prompt template ready.")

Prompt template ready.


## 4. Call the LLM + Parse JSON Output

This is the **full note generation** step — the main deliverable that removes the translation barrier. The output here is what the doctor will review and edit.

In [9]:
import re
import json

def call_natlas_llm(prompt: str, max_new_tokens: int = 300) -> str:
    """Generate text locally using the loaded N-ATLaS model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,   # low temp = consistent structured output
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the output, keep only what the model generated new
    generated = full_text[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):]
    return generated.strip()


def extract_json_from_text(text: str):
    """Pull a JSON object out of text that may include markdown fences or extra words."""
    cleaned = text.replace("```json", "").replace("```", "").strip()
    match = re.search(r"\{(?:[^{}]|(?:\{[^{}]*\}))*\}", cleaned, re.DOTALL)
    if match:
        return match.group(0)
    if cleaned.startswith("{"):
        return cleaned
    return None


def validate_structure(data: dict) -> dict:
    """Fill in any missing required fields."""
    required_fields = ["chief_complaint", "duration", "severity", "history","causes", "effect"]
    for field in required_fields:
        if field not in data or not data[field]:
            data[field] = "Not mentioned"
    return data


def parse_llm_output(raw_output: str) -> dict:
    """Extract, parse, and validate the JSON clinical note from raw LLM text."""
    json_str = extract_json_from_text(raw_output)
    if not json_str:
        return {
            "chief_complaint": "", "duration": "", "severity": "", "history": "",
            "_error": "No JSON found in output", "_raw": raw_output[:200]
        }
    try:
        data = json.loads(json_str)
        return validate_structure(data)
    except json.JSONDecodeError as e:
        return {
            "chief_complaint": "", "duration": "", "severity": "", "history": "",
            "_error": f"JSON parse failed: {e}", "_raw": json_str[:200]
        }


def structure_note(transcript: str) -> dict:
    """
    MAIN FUNCTION — the core deliverable.
    Takes raw ASR transcript text, returns structured English clinical note.
    """
    if not transcript or not transcript.strip():
        return {"chief_complaint": "", "duration": "", "severity": "", "history": "",
                "_error": "Empty transcript"}

    prompt = STRUCTURE_PROMPT.format(transcript=transcript)
    raw_output = call_natlas_llm(prompt)
    return parse_llm_output(raw_output)

print("LLM calling + JSON parsing functions ready.")

LLM calling + JSON parsing functions ready.


## 5. Keyword Extraction (Supporting Sidebar — NOT the Main Output)

This pulls out quick-reference clinical terms from the **raw transcript** so the doctor can scan them alongside the full note. This is supporting, not a replacement for the structured note above.

In [10]:
SYMPTOM_KEYWORDS = [
    "pain", "fever", "cough", "vomiting", "nausea", "headache", "dizziness",
    "fatigue", "weakness", "rash", "itching", "swelling", "difficulty breathing",
    "chest pain", "abdominal pain", "back pain", "joint pain"
]

DURATION_PATTERNS = [
    r"(\d+\s*(?:day|week|month|year|hour)s?)",
    r"(since\s+(?:yesterday|today|last\s+\w+))",
    r"(for\s+\d+\s*(?:day|week|month)s?)"
]

SEVERITY_KEYWORDS = [
    "mild", "moderate", "severe", "unbearable", "manageable",
    "very bad", "really bad", "not too bad", "little bit"
]

ANATOMICAL_SITES = [
    "head", "chest", "stomach", "abdomen", "back", "arm", "leg",
    "throat", "eye", "ear", "joint", "knee", "shoulder"
]

def extract_keywords(transcript: str) -> dict:
    """Extract clinical keywords from the raw transcript (supporting reference only)."""
    text = transcript.lower()
    extracted = {"symptoms": [], "duration": [], "severity": [], "anatomical_sites": [], "causes": [], "effect" : []}

    for symptom in SYMPTOM_KEYWORDS:
        if symptom in text:
            extracted["symptoms"].append(symptom)

    for pattern in DURATION_PATTERNS:
        extracted["duration"].extend(re.findall(pattern, text, re.IGNORECASE))

    for severity in SEVERITY_KEYWORDS:
        if severity in text:
            extracted["severity"].append(severity)

    for site in ANATOMICAL_SITES:
        if site in text:
            extracted["anatomical_sites"].append(site)

    for key in extracted:
        extracted[key] = list(set(extracted[key]))

    return extracted

print("Keyword extraction function ready.")

Keyword extraction function ready.


## 6. Combine Into One Patient Record

This bundles the full structured note (main) + keywords (supporting) + metadata into one record, ready for doctor review and database storage.

In [11]:
def build_patient_record(patient_id: str, language: str, transcript: str) -> dict:
    """
    Full pipeline: ASR transcript -> structured note + keywords -> patient record.
    This is what gets shown to the doctor for review, then saved to the database.
    """
    note = structure_note(transcript)              # MAIN deliverable
    keywords = extract_keywords(transcript)          # supporting reference

    record = {
        "patient_id": patient_id,
        "language": language,
        "raw_transcript": transcript,
        "chief_complaint": note.get("chief_complaint", ""),
        "duration": note.get("duration", ""),
        "severity": note.get("severity", ""),
        "history": note.get("history", ""),
        "causes": note.get("causes", ""),
        "effect": note.get("effect", ""),
        "extracted_keywords": keywords,
        "status": "ready_for_review"
    }
    return record

print("Patient record builder ready.")

Patient record builder ready.


## 7. Database Setup — SQLite

Each patient has a unique ID. Every visit is linked to that ID so the clinical history is traceable across multiple visits.

In [12]:
import sqlite3
from pathlib import Path

DB_PATH = Path("notes.db")   # In Colab this saves to the session storage
                              # For persistence, mount Google Drive and point here instead

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS patients (
            patient_id TEXT PRIMARY KEY,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS clinical_visits (
            visit_id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id TEXT NOT NULL,
            chief_complaint TEXT,
            duration TEXT,
            severity TEXT,
            history TEXT,
            causes TEXT,
            effect TEXT,
            language TEXT,
            extracted_keywords TEXT,
            raw_transcript TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
        )
    """)
    conn.commit()
    conn.close()

init_db()
print(f"Database initialized at {DB_PATH.resolve()}")

Database initialized at /content/notes.db


In [13]:
import time

def save_patient_record(record: dict, retries=5, delay=1):
    """Save a (doctor-reviewed) patient record to the database with retry logic for locks."""
    for i in range(retries):
        try:
            conn = sqlite3.connect(DB_PATH)

            conn.execute(
                "INSERT OR IGNORE INTO patients (patient_id) VALUES (?)",
                (record["patient_id"],)
            )

            conn.execute(
                """INSERT INTO clinical_visits
                   (patient_id, chief_complaint, duration, severity, history,causes, effect,
                    language, extracted_keywords, raw_transcript)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                (
                    record["patient_id"],
                    record["chief_complaint"],
                    record["duration"],
                    record["severity"],
                    record["history"],
                    record["causes"],
                    record["effect"],
                    record["language"],
                    json.dumps(record["extracted_keywords"]),
                    record["raw_transcript"],
                )
            )
            conn.commit()
            conn.close()
            print(f"Saved visit for patient {record['patient_id']}")
            return # Success, exit the function
        except sqlite3.OperationalError as e:
            if "database is locked" in str(e) and i < retries - 1:
                print(f"Database locked. Retrying in {delay} seconds... (Attempt {i+1}/{retries})")
                time.sleep(delay)
                delay *= 2 # Exponential backoff
            else:
                print(f"Error saving record for patient {record['patient_id']}: {e}")
                raise # Re-raise other operational errors or the last lock error


def get_patient_history(patient_id: str):
    """Retrieve all past visits for a patient, most recent first."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        """SELECT visit_id, chief_complaint, duration, severity, history,causes, effect,
                  language, created_at
           FROM clinical_visits
           WHERE patient_id = ?
           ORDER BY created_at DESC""",
        (patient_id,)
    )
    rows = cursor.fetchall()
    conn.close()
    return rows

print("Database save/retrieve functions ready.")

Database save/retrieve functions ready.


In [14]:
STRUCTURE_PROMPT = """You are a clinical note assistant specializing in multilingual healthcare.

Your task: Take a patient's raw speech (in Yoruba, Igbo, Hausa, or English)
and structure it into a clean, organized English clinical note.

INSTRUCTIONS:
1. If the patient spoke in Yoruba, Igbo, or Hausa, translate their meaning into clear English.
2. Extract and organize EXACTLY these six fields:
   - chief_complaint: The patient's main symptom or concern (1-2 sentences, in English)
   - duration: How long the problem has lasted (e.g., "since yesterday", "3 days", "1 week")
   - severity: How bad it is on a scale (mild, moderate, or severe)
   - history: Any other relevant details or secondary symptoms mentioned
   - causes: Anything the patient says may have caused or triggered the problem (e.g., an injury, food eaten, an activity, exposure to something)
   - effect: How the problem is affecting the patient's daily life or functioning (e.g., "can't sleep", "unable to work", "can't walk properly")

3. CRITICAL: Use ONLY information the patient actually said. Do NOT invent or assume details.
4. If a field is not mentioned by the patient, set it to "Not mentioned".
5. Respond with ONLY valid JSON. No explanatory text before or after the JSON.

EXAMPLE:
Patient speech: "My stomach has been hurting me since yesterday, I've been vomiting too, it's very bad. I think it's because of the food I ate at a party. I've not been able to go to work because of it."
Output:
{{
  "chief_complaint": "Abdominal pain with vomiting",
  "duration": "Since yesterday",
  "severity": "severe",
  "history": "Patient reports nausea accompanying the abdominal pain",
  "causes": "Patient suspects food eaten at a party",
  "effect": "Unable to go to work"
}}

Now process this patient's speech:
Patient speech: {transcript}

Output (JSON only, no other text):"""

# Sample transcripts (in practice these come from asr/transcribe.py output)
sample_transcripts = [
    {
        "patient_id": "P001",
        "language": "English",
        "transcript": "My stomach has been hurting me since yesterday, I've been vomiting too, it's very bad"
    },
    {
        "patient_id": "P002",
        "language": "Hausa",
        "transcript": "[Sample Hausa transcript here — replace with real ASR output]"
    },
]

for sample in sample_transcripts:
    pid = sample["patient_id"]
    lang = sample["language"]
    print(f"--- Processing {pid} ({lang}) ---")
    record = build_patient_record(
        patient_id=sample["patient_id"],
        language=sample["language"],
        transcript=sample["transcript"]
    )
    print(json.dumps(record, indent=2))
    print()

--- Processing P001 (English) ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{
  "patient_id": "P001",
  "language": "English",
  "raw_transcript": "My stomach has been hurting me since yesterday, I've been vomiting too, it's very bad",
  "chief_complaint": "Abdominal pain with vomiting",
  "duration": "Since yesterday",
  "severity": "severe",
  "history": "Patient reports nausea accompanying the abdominal pain",
  "causes": "Patient suspects food eaten at a party",
  "effect": "Unable to go to work",
  "extracted_keywords": {
    "symptoms": [
      "vomiting"
    ],
    "duration": [
      "since yesterday"
    ],
    "severity": [
      "very bad"
    ],
    "anatomical_sites": [
      "stomach"
    ],
    "causes": [],
    "effect": []
  },
  "status": "ready_for_review"
}

--- Processing P002 (Hausa) ---
{
  "patient_id": "P002",
  "language": "Hausa",
  "raw_transcript": "[Sample Hausa transcript here \u2014 replace with real ASR output]",
  "chief_complaint": "Not mentioned",
  "duration": "Not mentioned",
  "severity": "Not mentioned",
  "history": "No

## 8. End-to-End Test

Simulates the full pipeline: raw ASR transcript in → structured note + keywords → saved to database → retrieved back.

Replace the sample transcripts below with real ASR output once the ASR team's module is wired in.

In [15]:
# Review step simulation — in the real app, the doctor edits `record` here
# before it gets saved. For this test, we save as-is.

for sample in sample_transcripts:
    record = build_patient_record(
        patient_id=sample["patient_id"],
        language=sample["language"],
        transcript=sample["transcript"]
    )

    # Show the doctor what would be reviewed before saving
    print(f"--- Review for {record['patient_id']} ---")
    print(f"Chief complaint: {record['chief_complaint']}")
    print(f"Duration:        {record['duration']}")
    print(f"Severity:        {record['severity']}")
    print(f"History:         {record['history']}")
    print(f"Causes:          {record['causes']}")
    print(f"Effect:          {record['effect']}")
    print()

    save_patient_record(record)

--- Review for P001 ---
Chief complaint: Abdominal pain with vomiting
Duration:        Since yesterday
Severity:        severe
History:         Patient reports nausea accompanying the abdominal pain
Causes:          Patient suspects food eaten at a party
Effect:          Unable to go to work

Saved visit for patient P001
--- Review for P002 ---
Chief complaint: Not mentioned
Duration:        Not mentioned
Severity:        Not mentioned
History:         Not mentioned
Causes:          Not mentioned
Effect:          Not mentioned

Saved visit for patient P002


In [16]:
!lsof /content/notes.db

In [17]:
# Verify it was saved — pull back patient P001's history
history = get_patient_history("P001")
for visit in history:
    print(visit)

(1, 'Abdominal pain with vomiting', 'Since yesterday', 'severe', 'Patient reports nausea accompanying the abdominal pain', 'Patient suspects food eaten at a party', 'Unable to go to work', 'English', '2026-08-26 21:31:53')


## 9. Next Steps

- [ ] Replace sample transcripts with real ASR output from the ASR team's module
- [ ] Test with real Yoruba/Igbo/Hausa transcripts (ask team for samples)
- [ ] Tune `STRUCTURE_PROMPT` if output is inconsistent or hallucinating
- [ ] If persistence across sessions is needed, mount Google Drive and point `DB_PATH` there:
  ```python
  from google.colab import drive
  drive.mount('/content/drive')
  DB_PATH = Path('/content/drive/MyDrive/NCAIR-DSA/data/notes.db')
  ```
- [ ] Hand off `structure_note()`, `extract_keywords()`, `build_patient_record()`, `save_patient_record()` to the Frontend team for wiring into `app.py`
- [ ] Benchmark inference time per call (should ideally be under a few seconds)

In [18]:
from google.colab import drive
drive.mount('/content/drive')
DB_PATH = Path('/content/drive/MyDrive/NCAIR-DSA/data/notes.db')

Mounted at /content/drive


In [19]:
# 1. Clone the repository
!git clone https://github.com/Antasey/NCAIR-DSA-Group1

# 2. Move into the project directory
%cd NCAIR-DSA-Group1

# 3. Install all project dependencies
!pip install -r requirements.txt


Cloning into 'NCAIR-DSA-Group1'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 185 (delta 4), reused 0 (delta 0), pack-reused 166 (from 1)
Receiving objects: 100% (185/185), 159.25 KiB | 4.55 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/NCAIR-DSA-Group1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.7 MB/s eta 0:00:00


In [20]:
!git pull origin main

From https://github.com/Antasey/NCAIR-DSA-Group1
 * branch            main       -> FETCH_HEAD
Already up to date.


In [21]:
!ls


 app.py						  nlp
 asr						  README.md
 data						  requirements.txt
 env.example					  templates
 LICENSE					  tests
'NCAIR_DSA_LLM_Structuring_corrected (2).ipynb'


In [22]:
# Print the first 30 lines of app.py to verify it has your modifications
!head -n 30 app.py


"""
NCAIR-DSA — Gradio entry point.
Pipeline: audio capture -> ASR -> LLM structuring -> keyword extraction ->
review (individual fields editable, keywords as reference sidebar) -> patient record save.

Main goal: full structured English clinical note (removes translation barrier).
Keywords: supporting quick-reference sidebar for doctor during review.

Fields are kept SEPARATE (not one text blob) so what the doctor edits maps
directly to what gets saved in the database — no re-parsing free text.
"""
import gradio as gr
import json
# Mount Google Drive BEFORE importing clinical_note — that module checks
# whether Drive is mounted to decide where the database lives.
from asr.transcribe import transcribe_audio
from nlp.structure_note import structure_note
from nlp.extract_keywords import extract_keywords
from templates.clinical_note import init_db, save_patient_record as db_save_patient_record, get_patient_history

# Ensure database tables exist on startup
init_db()


def process_patient_

In [23]:
!pwd
!ls

/content/NCAIR-DSA-Group1
 app.py						  nlp
 asr						  README.md
 data						  requirements.txt
 env.example					  templates
 LICENSE					  tests
'NCAIR_DSA_LLM_Structuring_corrected (2).ipynb'


In [24]:
!ls templates/

clinical_note.py


In [25]:
!git log --all --full-history -- "*notes.db*"

commit b7c7aef30e3a24c3c1d83d21f3f8d8de8a143174
Author: Jesutomi Santa <104510167+Antasey@users.noreply.github.com>
Date:   Wed Aug 19 14:09:47 2026 +0100

    Implement keyword extraction and highlighting functions
    
    Added functions to extract and highlight clinical keywords from patient transcripts for structured notes.

commit 63d4761e6ec44286d9b7ff701733da4d07ea35b7
Author: Jesutomi Santa <104510167+Antasey@users.noreply.github.com>
Date:   Tue Aug 18 20:29:36 2026 +0100

    Create data/notes.db


In [26]:
# Safer non-interactive login (avoids the token being echoed into cell output/logs,
# which is what happened before — that leaked token must be revoked at
# https://huggingface.co/settings/tokens if you haven't already).
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face access token: ")
login(token=hf_token)
del hf_token

Enter your Hugging Face access token: ··········


In [27]:
!cat app.py

"""
NCAIR-DSA — Gradio entry point.
Pipeline: audio capture -> ASR -> LLM structuring -> keyword extraction ->
review (individual fields editable, keywords as reference sidebar) -> patient record save.

Main goal: full structured English clinical note (removes translation barrier).
Keywords: supporting quick-reference sidebar for doctor during review.

Fields are kept SEPARATE (not one text blob) so what the doctor edits maps
directly to what gets saved in the database — no re-parsing free text.
"""
import gradio as gr
import json
# Mount Google Drive BEFORE importing clinical_note — that module checks
# whether Drive is mounted to decide where the database lives.
from asr.transcribe import transcribe_audio
from nlp.structure_note import structure_note
from nlp.extract_keywords import extract_keywords
from templates.clinical_note import init_db, save_patient_record as db_save_patient_record, get_patient_history

# Ensure database tables exist on startup
init_db()


def process_patient_

In [36]:
import gradio as gr

# 1. Wrap everything inside the Blocks context manager
with gr.Blocks() as demo:

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### Full Clinical Note (Edit as Needed)")
            chief_complaint = gr.Textbox(label="Chief Complaint", lines=2, interactive=True)
            duration = gr.Textbox(label="Duration", lines=1, interactive=True)
            severity = gr.Textbox(label="Severity", lines=1, interactive=True)
            history = gr.Textbox(label="Relevant History", lines=3, interactive=True)
            causes = gr.Textbox(label="Causes", lines=2, interactive=True)
            effect = gr.Textbox(label="Effect on Daily Life", lines=2, interactive=True)

        with gr.Column(scale=1):
            gr.Markdown("### Quick Keywords (Reference Only)")
            keywords_display = gr.Textbox(
                label="For scanning during review — read only",
                lines=16,
                interactive=False,
                placeholder="Keywords will appear here..."
            ) # 2. Added the missing closing parenthesis here

# 3. Launch the app canvas
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://115b8ea3deafd54ae6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
%%writefile app.py
"""
NCAIR-DSA — Gradio entry point.
Pipeline: audio capture -> ASR -> LLM structuring -> keyword extraction ->
review (individual fields editable, keywords as reference sidebar) -> patient record save.

Main goal: full structured English clinical note (removes translation barrier).
Keywords: supporting quick-reference sidebar for doctor during review.

Fields are kept SEPARATE (not one text blob) so what the doctor edits maps
directly to what gets saved in the database — no re-parsing free text.
"""
import gradio as gr
import json
# Mount Google Drive BEFORE importing clinical_note — that module checks
# whether Drive is mounted to decide where the database lives.
from asr.transcribe import transcribe_audio
from nlp.structure_note import structure_note
from nlp.extract_keywords import extract_keywords
from templates.clinical_note import init_db, save_patient_record as db_save_patient_record, get_patient_history

# Ensure database tables exist on startup
init_db()


def process_patient_intake(patient_id, language, audio_file):
    """
    End-to-end pipeline:
    1. Transcribe audio to text
    2. Structure into full English clinical note (main deliverable) — separate fields
    3. Extract keywords from transcript (supporting reference)
    4. Return: individual editable fields + keywords sidebar + raw transcript (hidden state)

    Returns (in order matching Gradio outputs):
        chief_complaint, duration, severity, history, causes, effect,
        keywords_display_text, transcript_state, keywords_state, status_message
    """
    if not audio_file or not patient_id:
        return "", "", "", "", "", "", "", "{}", "", "Error: Patient ID and audio required"

    # --- Step 1: Transcribe ---
    try:
        transcript = transcribe_audio(audio_file, language)
    except Exception as e:
        return "", "", "", "", "", "", "", "{}", "", f"Error transcribing audio: {str(e)}"

    # --- Step 2: Structure into clinical note (MAIN deliverable) ---
    note = structure_note(transcript)

    # --- Step 3: Extract keywords from raw transcript (supporting) ---
    keywords = extract_keywords(transcript)

    if note.get("_error"):
        status = f"⚠ Note generated with issues: {note['_error']}"
    else:
        status = "✓ Ready for review"

    return (
        note.get("chief_complaint", ""),
        note.get("duration", ""),
        note.get("severity", ""),
        note.get("history", ""),
        note.get("causes", ""),
        note.get("effect", ""),
        json.dumps(keywords, indent=2), # Display keywords nicely
        transcript, # Raw transcript for internal state
        json.dumps(keywords), # Keywords for internal state
        status
    )

def handle_save_click(patient_id, language, chief_complaint, duration, severity, history, causes, effect, raw_transcript, extracted_keywords_json):
    """Callback to save the patient record when the save button is clicked."""
    if not patient_id or not chief_complaint:
        return "Error: Patient ID and Chief Complaint are required to save."

    record = {
        "patient_id": patient_id,
        "language": language,
        "chief_complaint": chief_complaint,
        "duration": duration,
        "severity": severity,
        "history": history,
        "causes": causes,
        "effect": effect,
        "raw_transcript": raw_transcript,
        "extracted_keywords": json.loads(extracted_keywords_json),
        "status": "doctor_reviewed"
    }
    try:
        db_save_patient_record(record)
        return "✔ Record saved successfully!"
    except Exception as e:
        return f"Error saving record: {str(e)}"

def handle_view_history_click(patient_id):
    """Callback to retrieve and display patient history."""
    if not patient_id:
        return "Error: Patient ID is required to view history."
    history = get_patient_history(patient_id)
    if not history:
        return "No history found for this patient."

    display_text = []
    for i, visit in enumerate(history):
        display_text.append(f"--- Visit {len(history) - i} ({visit[8]}) ---") # visit[8] is created_at
        display_text.append(f"  Chief Complaint: {visit[1]}")
        display_text.append(f"  Duration: {visit[2]}")
        display_text.append(f"  Severity: {visit[3]}")
        display_text.append(f"  History: {visit[4]}")
        display_text.append(f"  Causes: {visit[5]}")
        display_text.append(f"  Effect: {visit[6]}")
    return "\n".join(display_text)

# --- Gradio UI Definition ---

with gr.Blocks(title="NCAIR-DSA Clinical Note Structuring") as demo:
    gr.Markdown("# 🇳🇬 NCAIR-DSA Clinical Note Structuring")
    gr.Markdown("## ASR Text → LLM Structured Note + Keywords → Database")

    # Hidden state components for passing data between functions
    transcript_state = gr.State(value="")
    keywords_state = gr.State(value="{}")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Patient Info & Intake")
            patient_id = gr.Textbox(label="Patient ID", placeholder="e.g., P001, P002", interactive=True)
            language = gr.Dropdown(
                ["English", "Yoruba", "Igbo", "Hausa"],
                label="Patient Language",
                value="English"
            )
            audio_input = gr.Audio(type="filepath", label="Patient Speech (WAV or MP3)")
            transcribe_btn = gr.Button("Process Patient Intake", variant="primary")
            process_status = gr.Textbox(label="Processing Status", interactive=False)


        with gr.Column(scale=3):
            gr.Markdown("### Full Clinical Note (Edit as Needed)")
            chief_complaint = gr.Textbox(label="Chief Complaint", lines=2, interactive=True)
            duration = gr.Textbox(label="Duration", lines=1, interactive=True)
            severity = gr.Textbox(label="Severity", lines=1, interactive=True)
            history = gr.Textbox(label="Relevant History", lines=3, interactive=True)
            causes = gr.Textbox(label="Causes", lines=2, interactive=True)
            effect = gr.Textbox(label="Effect on Daily Life", lines=2, interactive=True)

        with gr.Column(scale=1):
            gr.Markdown("### Quick Keywords (Reference Only)")
            keywords_display = gr.Textbox(
                label="For scanning during review — read only",
                lines=16,
                interactive=False,
                placeholder="Keywords will appear here..."
            )

    # Step 3: Save
    gr.Markdown("## Step 3: Save to Patient History")
    save_btn = gr.Button("Save Patient Record to Database", variant="primary", size="lg")
    save_status = gr.Textbox(label="Save Status", interactive=False)

    # Bonus: View patient history
    gr.Markdown("## View Patient Clinical History")
    with gr.Row():
        history_btn = gr.Button("View History for This Patient ID")
    history_display = gr.Textbox(label="Past Visits", lines=8, interactive=False)

    # ------------------------------------------------------------------
    # Wire up the pipeline
    # ------------------------------------------------------------------

    transcribe_btn.click(
        fn=process_patient_intake,
        inputs=[patient_id, language, audio_input],
        outputs=[
            chief_complaint, duration, severity, history, causes, effect,
            keywords_display, transcript_state, keywords_state, process_status
        ]
    )

    save_btn.click(
        fn=handle_save_click,
        inputs=[
            patient_id, language, chief_complaint, duration, severity, history, causes, effect,
            transcript_state, keywords_state
        ],
        outputs=[save_status]
    )

    history_btn.click(
        fn=handle_view_history_click,
        inputs=[patient_id],
        outputs=[history_display]
    )

if __name__ == "__main__":
    # share=True gives a public https://xxxx.gradio.live link — needed since
    # this runs on Colab's server, not your own machine. Link lasts ~72 hours.
    demo.launch(share=True, debug=True)

Overwriting app.py


In [ ]:
!rm -f data/notes.db
!python app.py

Database ready at: /content/drive/MyDrive/NCAIR-DSA/data/notes.db
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
* Running on local URL:  http://127.0.0.1:7861
INFO:httpx:HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
* Running on public URL: https://57b67e162d3f0e81a3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-laun